In [0]:
print("===== GOLD LAYER VALIDATION =====")

dim_employee_df = spark.table(
    "databricks_project1.gold.dim_employee"
)

dim_facility_df = spark.table(
    "databricks_project1.gold.dim_facility"
)

dim_labor_df = spark.table(
    "databricks_project1.gold.dim_labor_position"
)

fact_df = spark.table(
    "databricks_project1.gold.fact_employee_payroll"
)

print("Dim Employee:", dim_employee_df.count())
print("Dim Facility:", dim_facility_df.count())
print("Dim Labor Position:", dim_labor_df.count())
print("Fact Employee Payroll:", fact_df.count())

verify Gold duplicate keys

In [0]:
from pyspark.sql import functions as F

print(
    "Duplicate Employee_Code in Fact:",
    fact_df
        .groupBy("Employee_Code")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    "Duplicate EmployeeKey in Dim Employee:",
    dim_employee_df
        .groupBy("EmployeeKey")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    "Duplicate Facility_Code in Dim Facility:",
    dim_facility_df
        .groupBy("Facility_Code")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    "Duplicate Labor_Position_Code in Dim Labor Position:",
    dim_labor_df
        .groupBy("Labor_Position_Code")
        .count()
        .filter(F.col("count") > 1)
        .count()
)


verify the known RI issue

In [0]:
fact_labor_missing = (
    fact_df
    .select("Labor_Position_Code")
    .dropDuplicates()
    .join(
        dim_labor_df
        .select("Labor_Position_Code")
        .dropDuplicates(),
        on="Labor_Position_Code",
        how="left_anti"
    )
)

print(
    "Missing Labor Position keys:",
    fact_labor_missing.count()
)

display(fact_labor_missing)